In [1]:
from lmms_eval.tasks.strokerehab.utils_primitives import load_strokerehab_primitives_dataset

In [2]:
ds = load_strokerehab_primitives_dataset(activity='RTT left side,RTT right side')
df = ds['test'].to_pandas()
control_mask = ~df['stroke']
first_rep_mask = df['path_v'].str.contains('RTT right side1') | df['path_v'].str.contains('RTT left side1')
df = df[control_mask & first_rep_mask]
patient_has_all_four = (df.groupby('patient')['id'].count() == 4)
df = df[df['patient'].isin(patient_has_all_four[patient_has_all_four].index.tolist())]
control_patient_list = "C00022,C00023,C00024,C00028,C00029".split(',')
df = df[df['patient'].isin(control_patient_list)].copy()

In [ ]:
import pandas as pd

test_set = True

ds = load_strokerehab_primitives_dataset(activity='RTT left side,RTT right side,shelf left side,shelf right side')
best_views = pd.read_csv('data/rs/best_views.txt')
df = ds['test'].to_pandas()

prompt_tune_patients = "C00020,C00023,S00019"
test_patients = (
    "C00022,C00024,C00025,C00028,C00029,"
    "S00013,S00016,S00023,S00010,S00012,"
    "S00011,S00017,S0001,S0003,S00018,"
    "S00021,S00029,S00031,S00034,S00049"
)

patients = test_patients if test_set else prompt_tune_patients
df = df[df['patient'].isin(patients.split(','))]

best_views_selected = pd.merge(df, best_views, on='id', how='inner')
df = (best_views_selected
        .sort_values('id').groupby(['patient', 'activity'])
        .first()
        .reset_index()
)
df["activity_type"] = df["activity"].str.extract(r"(RTT|shelf)", expand=False)
df = df.groupby(["patient", "activity_type"], as_index=False).first()
df = df.drop(columns="activity_type")

In [4]:
df['activity'].value_counts()

activity
RTT left side       10
shelf right side    10
RTT right side      10
shelf left side      8
Name: count, dtype: int64

In [5]:
len(df)

38

In [6]:
df['duration_s'].sum() / 60

38.620783333333335

In [7]:
import datasets
import pandas as pd

def load_strokerehab_final():
    # Same stroke patients
    ds = load_strokerehab_primitives_dataset(patients="S00013,S00016,S00023,S00011,S00017", reps="first")
    df_stroke = ds['test'].to_pandas()

    # C00026 -> C00027. Filter out double shelf/RTT exercises for control
    ds = load_strokerehab_primitives_dataset(patients="C00022,C00023,C00024,C00028,C00029", reps="first")
    df_control = ds['test'].to_pandas()
    df_control = df_control[~df_control['activity'].isin(['shelf left side', 'RTT left side'])]

    df_combined = pd.concat([df_stroke, df_control], ignore_index=True)

    dataset = datasets.Dataset.from_pandas(df_combined)
    dataset_dict = datasets.DatasetDict({'test': dataset})
    return dataset_dict

if __name__ == "__main__":
    ds = load_strokerehab_final()
    df = ds['test'].to_pandas()
    print(df['patient'].value_counts())
    print(df['activity'].value_counts())

patient
S00011    9
S00013    9
S00016    9
S00017    9
S00023    9
C00022    9
C00023    9
C00024    9
C00028    9
C00029    9
Name: count, dtype: int64
activity
brushing            10
combing             10
deodrant            10
glasses             10
drinking            10
face wash           10
feeding             10
RTT right side       9
shelf right side     9
RTT left side        1
shelf left side      1
Name: count, dtype: int64


In [8]:
from data.utils_strokerehab import DataPaths, PrimitiveLabelUtils
import os

primitives = {'idle': 0.0, 'reach': 0.0, 'transport': 0.0, 'reposition': 0.0, 'stabilize': 0.0}
for index, row in df.iterrows():
    path_l = os.path.join(DataPaths.RAW_LABEL_DIR, row['path_l'])
    action_seq = PrimitiveLabelUtils.convert_labels_to_action_sequence(path_l)
    for action in action_seq:
        primitives[action['action']] += action['duration']

In [9]:
action_seq[0]

{'action': 'idle', 'start_time': 0.0, 'duration': 1.35}

In [10]:
primitives

{'idle': 730.3009999999999,
 'reach': 873.3930000000001,
 'transport': 1984.3989999999983,
 'reposition': 522.3739999999999,
 'stabilize': 508.877}

In [11]:
from lmms_eval.tasks.strokerehab.utils_primitives import _get_primitives_score

In [12]:
def action_seq_to_action_list(path_l):
    path_l = os.path.join(DataPaths.RAW_LABEL_DIR, path_l)
    action_seq = PrimitiveLabelUtils.convert_labels_to_action_sequence(path_l)
    action_list = [action['action'] for action in action_seq]
    return action_list

from collections import deque

def action_seq_to_action_list_chunked(path_l, chunk_time=0.25):
    """Input: label CSV. Output: ONE primitive per window of chunk_time seconds."""
    path_l = os.path.join(DataPaths.RAW_LABEL_DIR, path_l)
    action_seq = PrimitiveLabelUtils.convert_labels_to_action_sequence(path_l)
    action_list = []

    q = deque()  # actions overlapping current window, in start_time order

    cur_start_time = 0.0
    cur_end_time = chunk_time

    i = 0  # pointer into action_seq (assumed sorted by start_time)
    last_end = action_seq[-1]['start_time'] + action_seq[-1]['duration']

    while cur_start_time < last_end:
        # Evict actions that no longer overlap [cur_start_time, cur_end_time)
        while q and (q[0]['start_time'] + q[0]['duration']) <= cur_start_time:
            q.popleft()

        # Add newly-overlapping actions (start before window end and end after window start)
        while i < len(action_seq) and action_seq[i]['start_time'] < cur_end_time:
            if (action_seq[i]['start_time'] + action_seq[i]['duration']) > cur_start_time:
                q.append(action_seq[i])
            i += 1

        # Choose label: one overlap → that primitive; multiple → the second primitive
        if len(q) > 0:
            window_start, window_end = cur_start_time, cur_end_time
            # For each overlapping action, compute the midpoint of its overlap with the window
            centers = []
            for idx, a in enumerate(q):
                a_start = a['start_time']
                a_end = a['start_time'] + a['duration']
                overlap_start = max(a_start, window_start)
                overlap_end = min(a_end, window_end)
                # Only consider if there's positive overlap (should be true by construction)
                if overlap_end > overlap_start:
                    center = 0.5 * (overlap_start + overlap_end)
                    centers.append((center, idx))
            if centers:
                centers.sort(key=lambda x: x[0])  # sort by overlap midpoint
                median_pos = (len(centers) - 1) // 2  # lower median for even counts
                _, chosen_idx_in_q = centers[median_pos]
                chosen = q[chosen_idx_in_q]['action']
                action_list.append(chosen)

        # Advance window
        cur_start_time += chunk_time
        cur_end_time += chunk_time

    return action_list


def most_common_prediction(gt_prims_chunked):
    return ['transport'] * len(gt_prims_chunked)

In [13]:
from lmms_eval.tasks.strokerehab.utils_primitives import _convert_motion_contact_to_primitives

def prim_to_motion_contact(prim):
    if prim == 'idle':
        return "No <SEP> No"
    elif prim == 'reach' or prim == 'reposition':
        return "Yes <SEP> No"
    elif prim == 'transport':
        return "Yes <SEP> Yes"
    elif prim == 'stabilize':
        return "No <SEP> Yes"
    else:
        raise ValueError("Unknown primitive: {prim}")

In [17]:
chunk_time = 8. / 15
# Ground truth
df['gt'] = df['path_l'].apply(lambda p: action_seq_to_action_list(p))
# Omniscient baseline
def create_omniscient_baseline(row):
    motion_contact = [prim_to_motion_contact(prim) for prim in row['gt_with_chunking']]
    times = [chunk_time * i for i in range(len(row['gt_with_chunking'])+1)]
    prims, _ = _convert_motion_contact_to_primitives(motion_contact, times)
    return prims

df['gt_with_chunking'] = df['path_l'].apply(lambda p: action_seq_to_action_list_chunked(p, chunk_time=chunk_time))

df['Omniscient Baseline'] = df.apply(create_omniscient_baseline, axis=1)


In [18]:
import numpy as np
import pandas as pd
from collections import Counter


def simulate_decomposed_sequence(motion_matrix_prob: pd.DataFrame,
                      grasp_matrix_prob: pd.DataFrame,
                      length: int,
                      ) -> list[str]:
    """
    Simulate a sequence of primitives based on transition probabilities.

    Args:
        motion_matrix_prob: DataFrame of shape (2, 2), rows sum to 1.
        grasp_matrix_prob: DataFrame of shape (2, 2), rows sum to 1.
        length: desired sequence length.
    """
    motion_states = [False, True]
    grasp_states = [False, True]
    seq = []
    curr_motion = False
    curr_grasp = False
    for _ in range(length):
        if curr_motion and curr_grasp:
            res  = "Yes <SEP> Yes"
        elif curr_motion and not curr_grasp:
            res = "Yes <SEP> No"
        elif not curr_motion and curr_grasp:
            res = "No <SEP> Yes"
        else:
            res = "No <SEP> No"
        seq.append(res)

        # Sample next states
        motion_probs = motion_matrix_prob.loc[curr_motion].to_numpy()
        grasp_probs = grasp_matrix_prob.loc[curr_grasp].to_numpy()

        if np.isnan(motion_probs).all() or motion_probs.sum() == 0:
            curr_motion = np.random.choice(motion_states)
        else:
            motion_probs = np.nan_to_num(motion_probs) / np.sum(np.nan_to_num(motion_probs))
            curr_motion = np.random.choice(motion_states, p=motion_probs)

        if np.isnan(grasp_probs).all() or grasp_probs.sum() == 0:
            curr_grasp = np.random.choice(grasp_states)
        else:
            grasp_probs = np.nan_to_num(grasp_probs) / np.sum(np.nan_to_num(grasp_probs))
            curr_grasp = np.random.choice(grasp_states, p=grasp_probs)
    
    times = [chunk_time * i for i in range(len(seq)+1)]

    prims, _ = _convert_motion_contact_to_primitives(seq, times)

    return prims


def simulate_direct_sequence(matrix_prob: pd.DataFrame, length: int, start=None) -> list[str]:
    """
    Simulate a sequence of primitives based on transition probabilities.

    Args:
        matrix_prob: DataFrame of shape (n, n), rows sum to 1.
        length: desired sequence length.
        start: optional starting primitive; if None, random.
    """
    prims = matrix_prob.index.tolist()
    if start is None:
        start = np.random.choice(prims)
    seq = [start]
    for _ in range(length - 1):
        curr = seq[-1]
        probs = matrix_prob.loc[curr].to_numpy()
        if np.isnan(probs).all() or probs.sum() == 0:
            next_prim = np.random.choice(prims)
        else:
            probs = np.nan_to_num(probs) / np.sum(np.nan_to_num(probs))
            next_prim = np.random.choice(prims, p=probs)
        seq.append(next_prim)
    return seq


# Transition matrix baseline
motion_pairs = []
grasp_pairs  = []
for seq in df['gt_with_chunking']:
    motion_seq = [True if prim in ['reach', 'transport', 'reposition'] else False for prim in seq]
    motion_pairs.extend(zip(motion_seq[:-1], motion_seq[1:]))
    grasp_seq = [True if prim in ['transport', 'stabilize'] else False for prim in seq]
    grasp_pairs.extend(zip(grasp_seq[:-1], grasp_seq[1:]))

motion_counts = Counter(motion_pairs)
grasp_counts  = Counter(grasp_pairs)

motion_matrix = pd.DataFrame(0, index=[False, True], columns=[False, True], dtype=int)
for (a, b), c in motion_counts.items():
    motion_matrix.loc[a, b] = c
motion_matrix_prob = motion_matrix.div(motion_matrix.sum(axis=1), axis=0)

grasp_matrix = pd.DataFrame(0, index=[False, True], columns=[False, True], dtype=int)
for (a, b), c in grasp_counts.items():
    grasp_matrix.loc[a, b] = c
grasp_matrix_prob = grasp_matrix.div(grasp_matrix.sum(axis=1), axis=0)


pairs = []
for seq in df['gt_with_chunking']:
    pairs.extend(zip(seq[:-1], seq[1:]))
counts = Counter(pairs)
prims = sorted({p for pair in counts for p in pair})
matrix = pd.DataFrame(0, index=prims, columns=prims, dtype=int)
for (a, b), c in counts.items():
    matrix.loc[a, b] = c
matrix_prob = matrix.div(matrix.sum(axis=1), axis=0)


df['Transition Matrix-based Baseline [DP]'] = [
    simulate_decomposed_sequence(motion_matrix_prob, grasp_matrix_prob, len(seq)) for seq in df['gt_with_chunking']
]

df['Transition Matrix-based Baseline [Direct]'] = [
    simulate_direct_sequence(matrix_prob, len(seq), start='idle') for seq in df['gt_with_chunking']
]

# Transport-only baseline
df['Transport-Only Baseline'] = df['Omniscient Baseline'].apply(lambda lst: ['transport' for _ in lst])


In [19]:
def row_to_metrics(row, pred_col):
    scores = _get_primitives_score(
        row[pred_col],
        row['gt'],
        dedupe=True
    )
    mae = scores['mae_reach'] + scores['mae_transport'] + scores['mae_reposition'] + scores['mae_stabilize'] + scores['mae_idle']
    count_truth = scores['count_truth']
    return {
        'edit_score': scores['edit_score'],
        'action_error_rate': scores['action_error_rate'],
        'relative_counting_error': mae / count_truth,
    }

In [20]:
cols = [
    'Transport-Only Baseline',
    'Transition Matrix-based Baseline [DP]',
    'Transition Matrix-based Baseline [Direct]',
    'Omniscient Baseline',
]

results = []

for col in cols:
    metrics = df.apply(lambda row: row_to_metrics(row, col), axis=1, result_type='expand')
    metrics['method'] = col
    results.append(metrics)

df_metrics = pd.concat(results, ignore_index=True)

In [21]:
grouped = (
    df_metrics
    .groupby("method")
    .agg(
        edit_score_mean=("edit_score", lambda x: np.mean(x)),
        edit_score_sem=("edit_score", lambda x: np.std(x, ddof=1) / np.sqrt(len(x))),
        aer_mean=("action_error_rate", lambda x: np.mean(x)),
        aer_sem=("action_error_rate", lambda x: np.std(x, ddof=1) / np.sqrt(len(x))),
        rce_mean=("relative_counting_error", lambda x: np.mean(x)),
        rce_sem=("relative_counting_error", lambda x: np.std(x, ddof=1) / np.sqrt(len(x))),
    )
    .reset_index()
)

In [22]:
grouped.sort_values('edit_score_mean', ascending=True)

,method,edit_score_mean,edit_score_sem,aer_mean,aer_sem,rce_mean,rce_sem
3,Transport-Only Baseline,3.655825,0.393673,0.963442,0.003937,0.975408,0.012219
1,Transition Matrix-based Baseline [DP],46.317382,0.911494,0.713323,0.075215,0.632803,0.077037
2,Transition Matrix-based Baseline [Direct],49.817966,1.038706,0.670664,0.085359,0.609202,0.086459
0,Omniscient Baseline,88.470100,0.865171,0.116448,0.008671,0.114993,0.008746


In [23]:
latex_table = (
    grouped
    .sort_values('edit_score_mean', ascending=True)
    .rename(columns={
        "method": "Model",
        "edit_score_mean": "ES ↑",
        "aer_mean": "AER ↓",
        "rce_mean": "RCE ↓",
        "edit_score_sem": "ES_sem",
        "aer_sem": "AER_sem",
        "rce_sem": "RCE_sem",
    })
    .assign(**{
        "ES ↑": lambda d: d.apply(lambda r: f"{r['ES ↑']:.2f} ± {r['ES_sem']:.2f}", axis=1),
        "AER ↓": lambda d: d.apply(lambda r: f"{r['AER ↓']:.2f} ± {r['AER_sem']:.2f}", axis=1),
        "RCE ↓": lambda d: d.apply(lambda r: f"{r['RCE ↓']:.2f} ± {r['RCE_sem']:.2f}", axis=1),
    })[["Model", "ES ↑", "AER ↓", "RCE ↓"]]
    .to_latex(index=False, escape=False, column_format="lccc")
)

print(latex_table)

\begin{tabular}{lccc}
\toprule
Model & ES ↑ & AER ↓ & RCE ↓ \\
\midrule
Transport-Only Baseline & 3.66 ± 0.39 & 0.96 ± 0.00 & 0.98 ± 0.01 \\
Transition Matrix-based Baseline [DP] & 46.32 ± 0.91 & 0.71 ± 0.08 & 0.63 ± 0.08 \\
Transition Matrix-based Baseline [Direct] & 49.82 ± 1.04 & 0.67 ± 0.09 & 0.61 ± 0.09 \\
Omniscient Baseline & 88.47 ± 0.87 & 0.12 ± 0.01 & 0.11 ± 0.01 \\
\bottomrule
\end{tabular}

